# circuit_agent

> Circuit agent with saccades, jumps, and Hebbian updates.

In [ ]:
#| default_exp circuit_agent

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
from dataclasses import dataclass
import numpy as np
from explRL.agents import Agent, Policy, RL
from explRL.circuit import outputs, compassStep, goalStep, jumpProb

In [ ]:
#| export
class circuitPolicy(Policy):
    "Goal weights wG (N,) and compass weights wC (N, N), all in [0, 1]."
    def __init__(self, wG, wC):
        super().__init__()
        wG, wC = np.asarray(wG, dtype=float), np.asarray(wC, dtype=float)
        if wG.ndim != 1 or wC.ndim != 2 or wC.shape[0] != wC.shape[1]:
            raise ValueError('wG must be (N,) and wC must be (N, N)')
        if wG.shape[0] != wC.shape[0]:
            raise ValueError('wG and wC must have the same N')
        if wG.min() < 0 or wG.max() > 1 or wC.min() < 0 or wC.max() > 1:
            raise ValueError('weights must lie in [0, 1]')
        self.wG, self.wC = wG.copy(), wC.copy()

@dataclass
class Saccade:
    "One saccade drawn from the policy."
    direction: int   # +1 CW (a up, c down), -1 CCW
    size: float      # radians (drawn in degrees, then converted)
    steps: int       # grid steps, at least 1

In [ ]:
#| export
class circuitAgent(Agent):
    "Agent with integer arena heading a and compass heading c on the nCirc grid."
    def __init__(self, a, policy, rl, params, rng=None):
        if not isinstance(policy, circuitPolicy):
            raise TypeError('policy must be a circuitPolicy instance')
        if len(policy.wG) != params.nCirc:
            raise ValueError('weights must have N = params.nCirc')
        super().__init__(None, None, policy, rl, params)
        self.a = int(a) % params.nCirc
        self.c = self.a
        self.rng = rng if rng is not None else np.random.default_rng()

    def sampleSaccade(self) -> Saccade:
        "Draw 1: direction (CW with probability dS at c); draw 2: lognormal size in degrees."
        p, N = self.params, self.params.nCirc
        dS = outputs(self.policy.wG, self.c, p)['dS']
        direction = 1 if self.rng.random() < dS else -1
        size = float(np.deg2rad(self.rng.lognormal(p.muS, p.sigmaS)))
        return Saccade(direction, size, max(1, int(math.ceil(size / (2 * np.pi / N)))))

    def move(self, d):
        "One grid step: arena heading moves by d, compass heading by -d (Q2)."
        N = self.params.nCirc
        self.a, self.c = (self.a + d) % N, (self.c - d) % N

    def maybeJump(self, act) -> bool:
        "Draw 3: jump the compass by N/2 with probability pJ (Eq. 27) given ring activity act."
        if self.rng.random() < jumpProb(self.policy.wC, act, self.c):
            self.c = (self.c + self.params.nCirc // 2) % self.params.nCirc
            return True
        return False

    def sampleFixation(self) -> float:
        "Draw 4: fixation length from IG(aF / nuF, aF^2 / sigF^2), nuF at the current c (Q5, Q6)."
        p = self.params
        nuF = outputs(self.policy.wG, self.c, p)['nuF']
        return float(self.rng.wald(p.aF / nuF, p.aF ** 2 / p.sigF ** 2))

In [ ]:
#| export
class CircuitRL(RL):
    "Hebbian learning of compass and goal weights, with switches for each update and for jumps."
    def __init__(self, policy, learnC=True, learnG=True, jumps=True):
        super().__init__(policy)
        self.learnC, self.learnG, self.jumps = bool(learnC), bool(learnG), bool(jumps)

    def chooseAction(self, agent):
        "Next saccade from the agent's policy."
        return agent.sampleSaccade()

    def updateCompass(self, agent, act, size):
        "One saccade grid step of anti-Hebbian compass learning (Eq. 38, Q7)."
        self.policy.wC = compassStep(self.policy.wC, act, agent.c, size, agent.params)

    def updateGoal(self, agent, R, nSteps):
        "nSteps dtG steps of reward-gated goal learning at the current compass heading (Eq. 39)."
        for _ in range(int(nSteps)):
            self.policy.wG = goalStep(self.policy.wG, agent.c, R, agent.params)

In [ ]:
from math import pi
from fastcore.test import test_eq, test_close, test_fail
from explRL.params import Params
from explRL.circuit import grid, sinGoal, outputs, jumpProb, ringActivity
from explRL.agents import flexiblePolicy

N = 32
p = Params()
wG = sinGoal(0.3, 0.0, N)
wC = np.ones((N, N)) * 0.5
pol = circuitPolicy(wG, wC)
test_fail(lambda: circuitPolicy(wG, wC[:16]), contains='(N, N)')
test_fail(lambda: circuitPolicy(wG + 2.0, wC), contains='[0, 1]')
small = circuitPolicy(wG[:16], np.ones((16, 16)) * 0.5)
test_fail(lambda: circuitAgent(0, small, CircuitRL(small), p), contains='nCirc')
test_fail(lambda: circuitAgent(0, flexiblePolicy(), CircuitRL(pol), p), contains='circuitPolicy')
ag = circuitAgent(37, pol, CircuitRL(pol), p, rng=np.random.default_rng(0))
test_eq((ag.a, ag.c), (5, 5))

Saccades, moves, jumps and fixations:

In [ ]:
sac = ag.sampleSaccade()
test_eq(type(sac), Saccade)
assert sac.steps >= 1 and 0 < sac.size < 2 * pi
test_eq(sac.steps, max(1, int(np.ceil(sac.size / (2 * pi / N)))))
a0, c0 = ag.a, ag.c
ag.move(1)
test_eq((ag.a, ag.c), ((a0 + 1) % N, (c0 - 1) % N))
ag.move(-1)
test_eq((ag.a, ag.c), (a0, c0))

act = ringActivity(ag.a, N, False)
always = np.zeros((N, N)); always[:, ag.c] = 1.0          # pJ = 1 at c
ag.policy.wC = always
assert ag.maybeJump(act)
test_eq(ag.c, (c0 + N // 2) % N)
never = np.zeros((N, N)); never[:, (ag.c + N // 2) % N] = 1.0   # pJ = 0 at c
ag.policy.wC = never
assert not ag.maybeJump(act)
assert ag.sampleFixation() > 0

**AC-6**: the CW rate over 20 000 draws matches `dS`; `pJ` is 0.5 for equal net weights.

In [ ]:
wG6 = sinGoal(0.4, grid(N)[4], N)
pol6 = circuitPolicy(wG6, wC)
ag6 = circuitAgent(0, pol6, CircuitRL(pol6), p, rng=np.random.default_rng(123))
dS = outputs(wG6, ag6.c, p)['dS']
cw = sum(ag6.sampleSaccade().direction > 0 for _ in range(20000))
test_close(cw / 20000, dS, eps=0.01)
test_close(jumpProb(np.ones((N, N)) * 0.3, act, 0), 0.5)

Each switch turns its update off; the updates change weights when on.

In [ ]:
pol2 = circuitPolicy(wG, wC.copy())
rl2 = CircuitRL(pol2, learnC=False, learnG=False, jumps=False)
ag2 = circuitAgent(3, pol2, rl2, p, rng=np.random.default_rng(1))
act2 = ringActivity(ag2.a, N, False)
rl_on = CircuitRL(pol2)
rl_on.updateCompass(ag2, act2, 0.5)
assert not np.allclose(pol2.wC, wC)
w_before = pol2.wG.copy()
rl_on.updateGoal(ag2, 1, 10)
assert not np.allclose(pol2.wG, w_before)
w_before = pol2.wG.copy()
rl_on.updateGoal(ag2, 0, 10)
test_close(pol2.wG, w_before)

from explRL.circuit_train import train_circuit
from explRL.envs import ZoneArena
wC_before, wG_before = pol2.wC.copy(), pol2.wG.copy()
h = train_circuit(ag2, ZoneArena(), Ttot=20.0)
test_close(pol2.wC, wC_before)
test_close(pol2.wG, wG_before)
assert not h.jumped.any()

Draws happen in a fixed order, so one seed gives the same draws:

In [ ]:
def run_draws(seed):
    pol = circuitPolicy(wG, wC.copy())
    ag = circuitAgent(2, pol, CircuitRL(pol), p, rng=np.random.default_rng(seed))
    sac = ag.sampleSaccade()
    ag.move(sac.direction)
    return sac.direction, sac.size, ag.maybeJump(ringActivity(ag.a, N, True)), ag.sampleFixation()
test_eq(run_draws(7), run_draws(7))
assert run_draws(7) != run_draws(8)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()